# Demo 3.5 — A bigger model is a sharper predictor
<a id="top"></a>

```yaml
title: Demo 3.5 — A bigger model is a sharper predictor
keywords: model scale, gpt2, probability mass, prediction quality, l13
estimated duration: 10
```

> **Lesson L01**, teacher demo 3.5. Written lecture [L0102_lecture.md](L0102_lecture.md) §4.
> Same next-word-prediction mechanism as Demo 1 — run across a **size ladder** of local models.

**Contents**

1. [Same prompt, three model sizes](#1-same-prompt-three-model-sizes)
2. [A harder seed](#2-a-harder-seed)
3. [Five more seeds across the ladder](#3-five-more-seeds-across-the-ladder)
4. [What this is — and is not](#4-what-this-is--and-is-not)

In [1]:
from __future__ import annotations

import torch
import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

transformers.logging.set_verbosity_error()  # quiet advisory HF logs in committed output

LADDER = ["gpt2", "gpt2-medium", "gpt2-large"]  # 124M -> 355M -> 774M
PARAMS = {"gpt2": "124M", "gpt2-medium": "355M", "gpt2-large": "774M"}

# Load the whole ladder once (downloads ~5 GB total on first run; cached afterwards).
MODELS = {name: GPT2LMHeadModel.from_pretrained(name).eval() for name in LADDER}
TOKENIZERS = {name: GPT2TokenizerFast.from_pretrained(name) for name in LADDER}


def top_k(name: str, prompt: str, k: int = 3) -> list[tuple[str, float]]:
    tok = TOKENIZERS[name]
    inputs = tok(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = MODELS[name](**inputs).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    return [(tok.decode([int(i)]), float(p)) for p, i in zip(top.values, top.indices, strict=True)]


def compare(prompt: str) -> None:
    print(f"prompt: {prompt!r}")
    for name in LADDER:
        pairs = ", ".join(f"{piece!r}={p:.3f}" for piece, p in top_k(name, prompt))
        print(f"  {name:12s} ({PARAMS[name]:>4}): {pairs}")

## 1. Same prompt, three model sizes

Watch the probability on the sensible continuation **concentrate** as the model grows, while
the tail of noise thins out.

In [2]:
compare("Water is made of hydrogen and")

prompt: 'Water is made of hydrogen and'
  gpt2         (124M): ' oxygen'=0.464, ' helium'=0.279, ' carbon'=0.034
  gpt2-medium  (355M): ' oxygen'=0.632, ' helium'=0.126, ' carbon'=0.089


  gpt2-large   (774M): ' oxygen'=0.839, ' carbon'=0.058, ' water'=0.021


` oxygen` climbs roughly **0.46 → 0.63 → 0.84** across 124M → 355M → 774M. Same loop, sharper
prediction.

[↑ Back to top](#top)

## 2. A harder seed

On a fact the small model barely knows, it spreads its bets across cities; the larger models
commit.

In [3]:
compare("The Eiffel Tower is located in the city of")

prompt: 'The Eiffel Tower is located in the city of'
  gpt2         (124M): ' Paris'=0.064, ' London'=0.046, ' Amsterdam'=0.034
  gpt2-medium  (355M): ' Paris'=0.691, ' Nice'=0.081, ' Lyon'=0.072
  gpt2-large   (774M): ' Paris'=0.597, ' E'=0.048, ' Lyon'=0.044


## 3. Five more seeds across the ladder

The same mechanism, five more prompts — each shows a different face of "bigger = sharper":

- **`The opposite of hot is`** — a fact all three know: the mass on ` cold` still climbs with size.
- **`Roses are red, violets are`** — the smallest model picks the *wrong* top word; scale corrects it to ` blue`.
- **`The capital of Japan is`** — the small model spreads its bets; the larger ones commit to ` Tokyo`.
- **`The chemical symbol for gold is`** — a capability *threshold*: only the 774M model surfaces ` Au`.
- **`The first man to walk on the moon was Neil`** — a counter-example: this is already easy, so all three are near-certain of ` Armstrong`.

In [4]:
compare("The opposite of hot is")
compare("Roses are red, violets are")
compare("The capital of Japan is")
compare("The chemical symbol for gold is")
compare("The first man to walk on the moon was Neil")

prompt: 'The opposite of hot is'


  gpt2         (124M): ' cold'=0.163, ' the'=0.082, ' a'=0.062
  gpt2-medium  (355M): ' cold'=0.740, ' cool'=0.066, ' not'=0.030
  gpt2-large   (774M): ' cold'=0.791, ' not'=0.048, ' cool'=0.043
prompt: 'Roses are red, violets are'
  gpt2         (124M): ' green'=0.242, ' blue'=0.210, ' white'=0.095


  gpt2-medium  (355M): ' blue'=0.744, ' green'=0.086, ' yellow'=0.046
  gpt2-large   (774M): ' blue'=0.773, ' yellow'=0.070, ' green'=0.050
prompt: 'The capital of Japan is'
  gpt2         (124M): ' the'=0.094, ' Tokyo'=0.067, ' home'=0.064
  gpt2-medium  (355M): ' Tokyo'=0.247, ' Osaka'=0.061, ' Kyoto'=0.049


  gpt2-large   (774M): ' Tokyo'=0.413, ' the'=0.054, ' also'=0.032
prompt: 'The chemical symbol for gold is'
  gpt2         (124M): ' the'=0.220, ' a'=0.082, ' "'=0.042
  gpt2-medium  (355M): ' G'=0.077, ' g'=0.066, ' the'=0.035
  gpt2-large   (774M): ' Au'=0.257, ' H'=0.032, ' O'=0.023
prompt: 'The first man to walk on the moon was Neil'
  gpt2         (124M): ' Armstrong'=0.961, ' de'=0.027, ' De'=0.002


  gpt2-medium  (355M): ' Armstrong'=0.997, ' A'=0.001, ' Young'=0.001
  gpt2-large   (774M): ' Armstrong'=0.996, ' A'=0.001, ' G'=0.000


Four of the five sharpen as the model grows; the fifth (` Armstrong`) was already a
near-certainty at 124M. Scale sharpens the predictions that have **room** to sharpen — it
doesn't manufacture confidence the small model already had.

[↑ Back to top](#top)

## 4. What this is — and is not

This is a **mechanistic** point: more capable = sharper distribution. It is one of two levers
on prediction quality; the other (the context *you* provide) is Demo 4.

It is **not** a model-selection lesson. *Which* model or provider you'd actually pick for a job —
a vision model for OCR, a strong reasoner for planning, a small fast model for cheap execution,
trading capability against cost, latency, and context length — is **L14 (choosing models &
providers)**. Here we only establish *that* capability sharpens prediction.

[↑ Back to top](#top)